# Witch Hat Atelier - Clasificador de signos (Colab + GPU)

Notebook **autocontenido**: no necesita los `.py` del repo. Reconoce que **signo/hechizo**
aparece en la imagen de un sello, usando *transfer learning* sobre **GPU gratuita**.

**Flujo:** subir recortes de referencia -> expansion sintetica -> entrenar -> evaluar -> predecir.

> Activa la GPU: menu **Entorno de ejecucion > Cambiar tipo de entorno > GPU (T4)**.

> Nota de lore: por defecto NO se voltea ni rota fuerte (espejar/invertir un signo
> cambia su significado en la obra).

## 1. Verificar GPU

In [ ]:
import torch
print("torch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU disponible:", torch.cuda.get_device_name(0))
else:
    print("SIN GPU. Ve a 'Entorno de ejecucion > Cambiar tipo de entorno > GPU' y reinicia.")

## 2. Subir tus recortes de referencia

Necesitas **1 o mas recortes por hechizo** (cuanto mas, mejor). Prepara un ZIP con una
carpeta por clase y subelo aqui:

```
refs/fuego_farolillo/ref.png
refs/nubes/ref1.png
refs/nubes/ref2.png
refs/escudo_de_hielo/ref.png
```

Alternativa (datasets grandes): monta Google Drive con
`from google.colab import drive; drive.mount('/content/drive')` y apunta `REFS` a tu carpeta.

In [ ]:
import zipfile, shutil
from pathlib import Path
from google.colab import files

DATA = Path('/content/data'); REFS = DATA/'refs'; RAW = DATA/'raw'
REFS.mkdir(parents=True, exist_ok=True)
IMG_EXT = {'.png','.jpg','.jpeg','.bmp','.webp','.tif','.tiff'}

up = files.upload()  # elige tu refs.zip
shutil.rmtree('/content/_unzip', ignore_errors=True)
for name in up:
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(name) as z:
            z.extractall('/content/_unzip')

# Busca la carpeta 'refs' dentro del zip; si no, usa la raiz extraida.
cands = list(Path('/content/_unzip').rglob('refs'))
src = cands[0] if cands else Path('/content/_unzip')
for cdir in sorted(p for p in src.iterdir() if p.is_dir()):
    dst = REFS/cdir.name; dst.mkdir(exist_ok=True)
    for p in cdir.iterdir():
        if p.suffix.lower() in IMG_EXT:
            shutil.copy(p, dst/p.name)

classes_found = sorted(d.name for d in REFS.iterdir() if d.is_dir())
print(f"{len(classes_found)} clases:", classes_found)

## 3. Expansion sintetica (offline)

De cada referencia genera `PER_CLASS` variantes (rotacion leve, escala, perspectiva,
grosor de trazo, desenfoque, brillo/contraste, ruido de papel, oclusiones).

In [ ]:
import random, numpy as np
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter

PER_CLASS = 80      # sube a 150+ si tienes pocas refs
SIZE      = 200
MAX_ROT   = 12.0
SEED      = 0

def perspective(img, mag, rng):
    w,h = img.size; d = mag*min(w,h); j = lambda: rng.uniform(-d,d)
    quad = (j(),j(), j(),h+j(), w+j(),h+j(), w+j(),j())
    return img.transform((w,h), Image.QUAD, quad, resample=Image.BILINEAR, fillcolor=255)

def scale_translate(img, rng, sr=(0.82,1.12), tr=0.06):
    w,h = img.size; s = rng.uniform(*sr); nw,nh = max(1,int(w*s)), max(1,int(h*s))
    r = img.resize((nw,nh), Image.BILINEAR); c = Image.new('L',(w,h),255)
    mdx,mdy = int(tr*w), int(tr*h)
    ox = (w-nw)//2 + rng.randint(-mdx,mdx); oy = (h-nh)//2 + rng.randint(-mdy,mdy)
    c.paste(r,(ox,oy)); return c

def vary_stroke(img, rng):
    r = rng.random()
    if r<0.33: return img.filter(ImageFilter.MinFilter(3))
    if r<0.66: return img.filter(ImageFilter.MaxFilter(3))
    return img

def add_noise(img, sigma):
    a = np.asarray(img).astype(np.float32) + np.random.normal(0,sigma,(img.size[1],img.size[0]))
    return Image.fromarray(np.clip(a,0,255).astype('uint8'))

def erase(img, rng, n=2):
    d = ImageDraw.Draw(img); w,h = img.size
    for _ in range(rng.randint(0,n)):
        ew,eh = rng.randint(w//20,w//6), rng.randint(h//20,h//6)
        x,y = rng.randint(0,w-ew), rng.randint(0,h-eh)
        d.rectangle([x,y,x+ew,y+eh], fill=rng.choice([255,248,240]))
    return img

def make_variant(ref, rng):
    im = ref.convert('L').resize((SIZE,SIZE), Image.BILINEAR)
    im = im.rotate(rng.uniform(-MAX_ROT,MAX_ROT), resample=Image.BILINEAR, fillcolor=255)
    im = perspective(im, rng.uniform(0,0.10), rng)
    im = scale_translate(im, rng)
    im = vary_stroke(im, rng)
    if rng.random()<0.4: im = im.filter(ImageFilter.GaussianBlur(rng.uniform(0.3,1.0)))
    im = ImageEnhance.Brightness(im).enhance(rng.uniform(0.9,1.1))
    im = ImageEnhance.Contrast(im).enhance(rng.uniform(0.85,1.2))
    im = add_noise(im, rng.uniform(2,10))
    im = erase(im, rng)
    return im.convert('RGB')

rng = random.Random(SEED); np.random.seed(SEED)
total = 0
for cdir in sorted(p for p in REFS.iterdir() if p.is_dir()):
    refs = [Image.open(p) for p in cdir.iterdir() if p.suffix.lower() in IMG_EXT]
    if not refs: continue
    dst = RAW/cdir.name; dst.mkdir(parents=True, exist_ok=True)
    for old in dst.glob('aug_*.png'): old.unlink()
    for i in range(PER_CLASS):
        make_variant(refs[i % len(refs)], rng).save(dst/f'aug_{i:04d}.png')
    total += PER_CLASS
    print(f'  {cdir.name:<24} {len(refs)} ref -> {PER_CLASS}')
print('Total imagenes generadas:', total)

## 4. Dataset, division train/val y aumentos

Reparto **estratificado** por clase. Aumentos en linea suaves (sin volteo).

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMAGENET_MEAN=[0.485,0.456,0.406]; IMAGENET_STD=[0.229,0.224,0.225]
IMG_SIZE=192; VAL_FRAC=0.2; BATCH=32; ALLOW_FLIP=False

def scan(raw, min_per=2):
    classes=[]
    for d in sorted(p for p in raw.iterdir() if p.is_dir()):
        imgs=[p for p in d.iterdir() if p.suffix.lower() in IMG_EXT]
        if len(imgs)>=min_per: classes.append((d.name,imgs))
    names=[c[0] for c in classes]
    samples=[(p,i) for i,(_,imgs) in enumerate(classes) for p in imgs]
    return samples, names

def split(samples, vf, seed):
    by={}; [by.setdefault(y,[]).append(p) for p,y in samples]
    rng=random.Random(seed); tr=[]; va=[]
    for y,ps in by.items():
        ps=ps[:]; rng.shuffle(ps)
        nv=max(1,round(len(ps)*vf)) if len(ps)>=2 else 0
        va += [(p,y) for p in ps[:nv]]; tr += [(p,y) for p in ps[nv:]]
    rng.shuffle(tr); rng.shuffle(va); return tr,va

class DS(Dataset):
    def __init__(s,samp,tf): s.samp=samp; s.tf=tf
    def __len__(s): return len(s.samp)
    def __getitem__(s,i):
        p,y=s.samp[i]; return s.tf(Image.open(p).convert('RGB')), y

norm=transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)
ops=[transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.RandomRotation(12,fill=255),
     transforms.RandomAffine(0,translate=(0.06,0.06),scale=(0.9,1.1),fill=255),
     transforms.ColorJitter(0.2,0.2), transforms.RandomPerspective(0.15,0.3,fill=255)]
if ALLOW_FLIP: ops.insert(1, transforms.RandomHorizontalFlip())
tf_train=transforms.Compose(ops+[transforms.ToTensor(),norm])
tf_val=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.ToTensor(),norm])

samples, CLASSES = scan(RAW)
assert len(CLASSES)>=2, 'Necesitas >=2 clases con >=2 imagenes.'
tr,va = split(samples, VAL_FRAC, 42)
print(f'{len(CLASSES)} clases | train {len(tr)} | val {len(va)}')
train_loader=DataLoader(DS(tr,tf_train),batch_size=BATCH,shuffle=True,num_workers=2,pin_memory=True)
val_loader  =DataLoader(DS(va,tf_val),batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True)

## 5. Modelo y entrenamiento (GPU + mixed precision)

`resnet18` preentrenado, *label smoothing*, *cosine schedule* y AMP. Guarda el mejor modelo.

In [ ]:
import torch.nn as nn
from torchvision import models

EPOCHS=40; LR=1e-3; ARCH='resnet18'; FULL_FINETUNE=False
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def build(arch,n,freeze):
    if arch=='resnet18':
        net=models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        if freeze:
            for p in net.parameters(): p.requires_grad=False
        net.fc=nn.Linear(net.fc.in_features,n)
    else:
        net=models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        if freeze:
            for p in net.features.parameters(): p.requires_grad=False
        net.classifier[3]=nn.Linear(net.classifier[3].in_features,n)
    return net

net=build(ARCH,len(CLASSES),freeze=not FULL_FINETUNE).to(device)
counts=np.bincount([y for _,y in tr],minlength=len(CLASSES))
w=torch.tensor(counts.sum()/np.maximum(counts,1),dtype=torch.float32); w=w/w.mean()
crit=nn.CrossEntropyLoss(weight=w.to(device),label_smoothing=0.1)
opt=torch.optim.Adam([p for p in net.parameters() if p.requires_grad],lr=LR,weight_decay=1e-4)
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
scaler=torch.cuda.amp.GradScaler(enabled=device.type=='cuda')

def run(loader,train):
    net.train(train); tot=cor=0; ls=0.0
    torch.set_grad_enabled(train)
    for x,y in loader:
        x,y=x.to(device),y.to(device)
        if train: opt.zero_grad()
        with torch.autocast(device_type=device.type,enabled=device.type=='cuda'):
            out=net(x); loss=crit(out,y)
        if train:
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        ls+=loss.item()*x.size(0); cor+=(out.argmax(1)==y).sum().item(); tot+=x.size(0)
    torch.set_grad_enabled(True); return ls/max(tot,1), cor/max(tot,1)

best=-1; best_state=None
for ep in range(1,EPOCHS+1):
    trl,tra=run(train_loader,True); val,vaa=run(val_loader,False); sched.step()
    mark=''
    if vaa>=best: best=vaa; best_state={k:v.cpu().clone() for k,v in net.state_dict().items()}; mark='  *'
    print(f'Epoch {ep:3d}/{EPOCHS} | train {trl:.3f}/{tra:.3f} | val {val:.3f}/{vaa:.3f}{mark}')
print('Mejor val acc:', round(best,3))

## 6. Reporte por clase y matriz de confusion

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
net.load_state_dict(best_state); net.eval()
ys=[]; ps=[]
with torch.no_grad():
    for x,y in val_loader:
        ps += net(x.to(device)).argmax(1).cpu().tolist(); ys += y.tolist()
print(classification_report(ys,ps,labels=list(range(len(CLASSES))),target_names=CLASSES,zero_division=0))
print('Matriz de confusion (filas=real, col=pred):')
print(confusion_matrix(ys,ps,labels=list(range(len(CLASSES)))))

## 7. Guardar el modelo (descarga + Drive opcional)

In [ ]:
from pathlib import Path
import json
MODELS=Path('/content/models'); MODELS.mkdir(exist_ok=True)
ckpt={'state_dict':best_state,'classes':CLASSES,'arch':ARCH,'img_size':IMG_SIZE,
      'normalize_imagenet':True,'best_val_acc':best}
torch.save(ckpt, MODELS/'best.pt')
(MODELS/'classes.json').write_text(json.dumps(CLASSES,indent=2))
print('Guardado en', MODELS/'best.pt')

from google.colab import files
files.download(str(MODELS/'best.pt'))   # descarga a tu PC
# Para Drive:
# from google.colab import drive; drive.mount('/content/drive')
# import shutil; shutil.copy(MODELS/'best.pt','/content/drive/MyDrive/best.pt')

## 8. Predecir un sello nuevo

In [ ]:
import torch.nn.functional as F
from google.colab import files
up=files.upload()
tf_test=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.ToTensor(),norm])
net.eval()
for name in up:
    img=Image.open(name).convert('RGB'); x=tf_test(img).unsqueeze(0).to(device)
    with torch.no_grad(): prob=F.softmax(net(x),1)[0]
    k=min(5,len(CLASSES)); conf,idx=prob.topk(k)
    print(f'\n{name}')
    for r,(c,i) in enumerate(zip(conf,idx),1):
        print(f'  {r}. {CLASSES[i]:<22} {float(c)*100:5.1f}%')